In [ ]:
import pandas as pd
import sys
from pathlib import Path
from datetime import date
diretorio_nb = Path.cwd()
sys.path.insert(0, str(diretorio_nb))
from trata_dados import salario_internacional_simples, taxa_media_inflacao
caminho_csv = diretorio_nb / 'jobs_in_data.csv'
df = pd.read_csv(caminho_csv)

In [ ]:
nomes_colunas = {
    'work_year': 'ano_trabalho',
    'job_title': 'cargo',
    'job_category': 'categoria_cargo',
    'salary_currency': 'moeda_salario',
    'salary': 'salario',
    'salary_in_usd': 'salario_dolar',
    'employee_residence': 'residencia_funcionario',
    'experience_level': 'nivel_experiencia',
    'employment_type': 'tipo_contrato',
    'work_setting': 'modelo_trabalho',
    'company_location': 'localizacao_empresa',
    'company_size': 'porte_empresa'
}
df_pt_br = df.rename(columns=nomes_colunas)

In [3]:
df_pt_br['ano_trabalho'] = pd.to_datetime(df_pt_br['ano_trabalho'], format='%Y')

In [ ]:
# Cache simples: lista de dicionários
# cada entrada: {'chave': 'pais ano', 'pais': ..., 'ano': ..., 'taxa': ..., 'fator': ...}
cache_inflacao = []

def encontra_chave_exata(chave):
    for entrada in cache_inflacao:
        if entrada['chave'] == chave:
            return entrada
    return None

def encontra_pais(pais):
    for entrada in cache_inflacao:
        if entrada['pais'] == pais:
            return entrada
    return None

ano_atual = date.today().year
resultados = []

for indice, linha in df_pt_br.iterrows():
    try:
        # país: preferir residência do funcionário, senão localização da empresa
        pais = linha.get('residencia_funcionario') or linha.get('localizacao_empresa')
        if pd.isna(pais):
            resultados.append(None)
            continue
        # ano original
        valor_ano = linha['ano_trabalho']
        if hasattr(valor_ano, 'year'):
            ano = int(valor_ano.year)
        else:
            ano = int(valor_ano)
        chave = f'{pais} {ano}'

        entrada = encontra_chave_exata(chave)
        if entrada is not None:
            taxa_media = entrada['taxa']
            valor_fator = entrada['fator']
        else:
            entrada_pais = encontra_pais(pais)
            if entrada_pais is not None:
                # reutiliza fator já obtido para o país (evita nova requisição do PPC)
                valor_fator = entrada_pais['fator']
                # ainda precisamos da inflação para o intervalo específico
                taxa_media = taxa_media_inflacao(pais, ano, ano_atual)
                cache_inflacao.append({'chave': chave, 'pais': pais, 'ano': ano, 'taxa': taxa_media, 'fator': valor_fator})
            else:
                # obtém inflação média
                taxa_media = taxa_media_inflacao(pais, ano, ano_atual)
                # obtém fator PPC e salário convertido temporário (apenas para pegar o fator)
                salario_temp_conv, valor_fator = salario_internacional_simples(pais, 1.0, ano=ano_atual)
                cache_inflacao.append({'chave': chave, 'pais': pais, 'ano': ano, 'taxa': taxa_media, 'fator': valor_fator})

        salario_local = linha.get('salario')
        if pd.isna(salario_local):
            resultados.append(None)
            continue

        anos_diff = ano_atual - ano
        if anos_diff <= 0:
            salario_corrigido = salario_local
        else:
            # aplica correção composta usando a taxa média anual
            salario_corrigido = salario_local * (1 + taxa_media / 100) ** anos_diff

        # converte para USD usando o fator (função retorna lista [convertido, fator])
        salario_convertido, fator_usado = salario_internacional_simples(pais, salario_corrigido, ano=ano_atual)

        resultados.append(salario_convertido)
    except Exception:
        resultados.append(None)

df_pt_br['salario_internacional_atual'] = resultados

In [ ]:
df_pt_br[['salario','salario_internacional_atual']].head()

In [ ]:
# Exibir cache usado (país e ano -> taxa média e fator)
cache_inflacao[:20]